In [ ]:
# Parameters — override these to run on a different dataset
DATASET_PATH = "../top100/top100_measures.jsonl"
DATASET_NAME = "Top100"
RAMA_FIELD = "rama3"  # rama3 for top100, rama4 for top500, rama_category for top8000/top2018

In [ ]:
import json
import warnings
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from collections import Counter, defaultdict
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=RuntimeWarning)
plt.rcParams.update({"figure.dpi": 150, "font.size": 9, "axes.titlesize": 11, "figure.facecolor": "white"})

In [ ]:
with open(DATASET_PATH) as f:
    records = [json.loads(line) for line in f
               if not json.loads(line).get("_meta")]

n_files = len(set(r["file"] for r in records))
n_chains = len(set((r["file"], r.get("chain", "")) for r in records))

display(Markdown(f"""# {DATASET_NAME} Dataset — General Statistics and Geometric Analysis

**Residues:** {len(records):,} | **Chains:** {n_chains:,} | **Structures:** {n_files:,}
| **Source:** {DATASET_PATH.split('/')[-1]}
| **Generated by:** pydangle-biopython v0.5.1

This document provides the following analyses:

1. **General Information** — dataset overview, measurements available, and summary statistics
2. **Amino Acid Composition and Chain Length Distribution** — comparison with UniProt and PDB reference databases
3. **Torsion Angle Distributions** — univariate (backbone, omega deviation, sidechain) and multivariate (Ramachandran) distributions using circular statistics
4. **Bond Angle Distributions** — tau (N−Cα−C) and other non-periodic angles using linear statistics
5. **References**
"""))

In [ ]:
def extract(field):
    return np.array([r[field] for r in records if r.get(field) is not None])

phi = extract("phi")
psi = extract("psi")
omega = extract("omega")
tau = extract("tau")

chi_fields = ["chi1", "chi2", "chi3", "chi4"]
chi = {f: extract(f) for f in chi_fields}
chi = {f: v for f, v in chi.items() if len(v) > 0}
rama = np.array([r.get("rama_category", "Unknown") for r in records])

def circ_stats(angles):
    n = len(angles)
    s = np.sum(np.sin(np.deg2rad(angles)))
    c = np.sum(np.cos(np.deg2rad(angles)))
    mean = np.degrees(np.arctan2(s / n, c / n))
    R = np.sqrt((s / n)**2 + (c / n)**2)
    std = np.degrees(np.sqrt(-2 * np.log(R))) if R > 1e-15 else 180.0
    return mean, std, n

chi_summary = ", ".join(f"{f}: {len(v):,}" for f, v in chi.items()) if chi else "none"
files = set(r["file"] for r in records)
chains = set((r["file"], r.get("chain", "")) for r in records)
rama_counts = Counter(r.get("rama_category") for r in records if r.get("rama_category"))

display(Markdown(f"""## 1. General Information

| | |
|---|---|
| **Dataset** | {DATASET_NAME} |
| **Total residues** | {len(records):,} |
| **Unique structures** | {len(files):,} |
| **Unique chains** | {len(chains):,} |

**Measurements available:** phi: {len(phi):,}, psi: {len(psi):,},
omega: {len(omega):,}, tau: {len(tau):,}, {chi_summary}.
Not all residues have all measurements — terminal residues lack phi/psi/omega,
and glycine and alanine lack sidechain chi angles.
"""))

print("Ramachandran category distribution:")
for cat, n in rama_counts.most_common():
    print(f"  {cat:<12s} {n:>8,}  ({100*n/len(records):5.2f}%)")
print()
print("Torsion angles (circular statistics):")
print(f"{'  Angle':<10s} {'N':>10s} {'Circ Mean':>12s} {'Circ Std':>12s}")
print(f"  {'-'*8:<8s} {'-'*10:>10s} {'-'*12:>12s} {'-'*12:>12s}")
for name, vals in [("phi", phi), ("psi", psi), ("omega", omega)]:
    m, s, n = circ_stats(vals)
    print(f"  {name:<8s} {n:>10,} {m:>12.3f} {s:>12.3f}")
for name, vals in chi.items():
    m, s, n = circ_stats(vals)
    print(f"  {name:<8s} {n:>10,} {m:>12.3f} {s:>12.3f}")
print()
print("Bond angles (linear statistics):")
print(f"  {'tau':<8s} {len(tau):>10,} {tau.mean():>12.3f} {tau.std():>12.3f}")

In [ ]:
NONSTANDARD_MAPPING = {"MSE": "MET"}
UNIPROT_PCT = {"ALA": 8.25, "ARG": 5.53, "ASN": 4.06, "ASP": 5.45, "CYS": 1.37, "GLN": 3.93, "GLU": 6.75, "GLY": 7.07, "HIS": 2.27, "ILE": 5.96, "LEU": 9.66, "LYS": 5.84, "MET": 2.42, "PHE": 3.86, "PRO": 4.73, "SER": 6.56, "THR": 5.34, "TRP": 1.08, "TYR": 2.92, "VAL": 6.87}

# Compute deviations to decide whether to flag them
resname_counts_tmp = Counter()
for r in records:
    rn = NONSTANDARD_MAPPING.get(r["resname"], r["resname"])
    resname_counts_tmp[rn] += 1
total_tmp = sum(resname_counts_tmp.values())
max_delta = max(abs(100 * resname_counts_tmp.get(aa, 0) / total_tmp - up)
                for aa, up in UNIPROT_PCT.items())

filtering_note = ""
if max_delta > 2.0:
    # Identify the largest deviations
    deltas = [(aa, 100 * resname_counts_tmp.get(aa, 0) / total_tmp - up)
              for aa, up in UNIPROT_PCT.items()]
    deltas.sort(key=lambda x: abs(x[1]), reverse=True)
    top3 = deltas[:3]
    examples = "; ".join(f"{aa} ({d:+.1f}%)" for aa, d in top3)
    filtering_note = (
        f"\n\n**Note:** This dataset shows substantial deviations from reference "
        f"amino acid frequencies (e.g., {examples}). "
        f"This is expected for datasets with stringent quality filtering, which "
        f"preferentially retains well-ordered core residues (enriching Ala, Val, Gly) "
        f"and removes disordered surface residues (depleting Lys, Glu, Arg). "
        f"A detailed analysis of these filtering effects is deferred to future work."
    )

display(Markdown(f"""## 2. Amino Acid Composition and Chain Length Distribution

### 2.1 Representativeness

The figure and table below compare the {DATASET_NAME} amino acid frequencies against two
reference databases:

- **UniProt/SwissProt** (~2024): amino acid frequencies across all reviewed protein sequences,
  representing the broadest available view of protein sequence space.
- **PDB X-ray** (~2025): frequencies from a 10,000-entity sample of X-ray crystallographic
  structures in the RCSB Protein Data Bank.

The two reference distributions are highly similar (most amino acids differ by <0.5%),
indicating that the PDB's well-known crystallization bias — overrepresentation of soluble,
globular, well-expressing proteins from model organisms — has only a modest effect on
overall amino acid composition.

The {DATASET_NAME} dataset ({len(records):,} residues from {n_files:,}
structures) shows additional deviations from both references due to
quality filtering, homology reduction, and sample size effects.{filtering_note}
"""))

In [ ]:
PDB_XRAY_PCT = {"ALA": 8.05, "ARG": 4.83, "ASN": 4.48, "ASP": 5.75, "CYS": 1.55, "GLN": 3.81, "GLU": 6.20, "GLY": 7.69, "HIS": 2.39, "ILE": 5.31, "LEU": 8.61, "LYS": 6.05, "MET": 2.16, "PHE": 3.93, "PRO": 4.73, "SER": 6.22, "THR": 5.90, "TRP": 1.52, "TYR": 3.67, "VAL": 6.90}

resname_counts = resname_counts_tmp
sorted_aa = resname_counts.most_common()
names = [aa for aa, _ in sorted_aa]
counts_aa = [c for _, c in sorted_aa]
total = total_tmp
pcts = [100 * c / total for c in counts_aa]
uniprot_vals = [UNIPROT_PCT.get(aa, 0) for aa in names]
pdb_vals = [PDB_XRAY_PCT.get(aa, 0) for aa in names]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(names))
bars = ax.bar(x, pcts, 0.6, color="#4393c3", alpha=0.85, label=f"{DATASET_NAME}", edgecolor="white", linewidth=0.3)
ax.plot(x, uniprot_vals, "o-", color="#d95f02", markersize=5, linewidth=1.5, alpha=0.8, label="UniProt/SwissProt", zorder=5)
ax.plot(x, pdb_vals, "s--", color="#7570b3", markersize=4, linewidth=1.2, alpha=0.7, label="PDB X-ray", zorder=5)
for bar, pct in zip(bars, pcts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15, f"{pct:.1f}", ha="center", va="bottom", fontsize=5.5, color="#333")
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=8)
ax.set_ylabel("Percent (%)", fontsize=10)
ax.set_title(f"{DATASET_NAME} \u2014 Amino Acid Composition vs. Reference Databases", fontsize=12, fontweight="bold")
ax.legend(fontsize=9, loc="upper right")
ax.axhline(5.0, color="gray", linewidth=0.5, linestyle=":", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"{'Residue':<8s} {'Count':>8s} {'Dataset%':>9s} {'UniProt%':>9s} {'PDB%':>9s} {'\u0394 UniProt':>10s}")
print(f"{'-'*8:<8s} {'-'*8:>8s} {'-'*9:>9s} {'-'*9:>9s} {'-'*9:>9s} {'-'*10:>10s}")
for aa, c in sorted_aa:
    dp = 100 * c / total
    up = UNIPROT_PCT.get(aa, 0)
    pp = PDB_XRAY_PCT.get(aa, 0)
    delta = dp - up
    print(f"{aa:<8s} {c:>8,} {dp:>8.2f}% {up:>8.2f}% {pp:>8.2f}% {delta:>+9.2f}%")

In [ ]:
display(Markdown("### 2.2 Chain Length Distribution"))

chain_lengths = defaultdict(int)
for r in records:
    key = (r["file"], r.get("chain", ""))
    chain_lengths[key] += 1
lengths = np.array(list(chain_lengths.values()))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.hist(lengths, bins=60, range=(0, max(600, np.percentile(lengths, 99))), color="#7570b3", alpha=0.8, edgecolor="white", linewidth=0.3)
ax1.axvline(np.median(lengths), color="red", linewidth=1, linestyle="--", alpha=0.7, label=f"Median = {np.median(lengths):.0f}")
ax1.axvline(np.mean(lengths), color="orange", linewidth=1, linestyle="--", alpha=0.7, label=f"Mean = {np.mean(lengths):.0f}")
ax1.set_xlabel("Residues per chain", fontsize=10)
ax1.set_ylabel("Count", fontsize=10)
ax1.set_title(f"{DATASET_NAME} — Chain Length Distribution", fontsize=12, fontweight="bold")
ax1.legend(fontsize=8)
ax2.boxplot(lengths, vert=True, widths=0.5, patch_artist=True, boxprops=dict(facecolor="#7570b3", alpha=0.5), medianprops=dict(color="red", linewidth=1.5))
ax2.set_ylabel("Residues per chain", fontsize=10)
ax2.set_title("Chain Length Summary", fontsize=12, fontweight="bold")
ax2.set_xticklabels([DATASET_NAME])
plt.tight_layout()
plt.show()

print(f"Chain length statistics ({len(lengths):,} chains):")
print(f"  Min:      {lengths.min():>8,}")
print(f"  Q1:       {np.percentile(lengths, 25):>8.0f}")
print(f"  Median:   {np.median(lengths):>8.0f}")
print(f"  Mean:     {np.mean(lengths):>8.1f}")
print(f"  Q3:       {np.percentile(lengths, 75):>8.0f}")
print(f"  Max:      {lengths.max():>8,}")
print(f"  Std:      {np.std(lengths):>8.1f}")

In [ ]:
display(Markdown("""## 3. Torsion Angle Distributions (Circular Statistics)

Torsion angles (φ, ψ, ω, χ) are periodic variables defined on the interval
[−180°, +180°]. Standard (linear) arithmetic means and standard deviations
give misleading results when values cluster near the ±180° boundary — for
example, a linear mean of trans peptide bond ω values near +179° and −179°
yields ~0° rather than the correct ~180°. All torsion angle summary statistics
in this report use **circular statistics**: the circular mean is computed as
atan2(⟨sin θ⟩, ⟨cos θ⟩), and the circular standard deviation as √(−2 ln R̄)
where R̄ is the mean resultant length.

### 3.1 Univariate Distributions
"""))

In [ ]:
def rose_plot(ax, angles_deg, n_bins=36, title="", color="steelblue", alpha=0.8):
    """Draw a rose plot (circular histogram) on a polar axes."""
    bin_width = 2 * np.pi / n_bins
    rads = np.deg2rad(angles_deg) % (2 * np.pi)
    bins = np.linspace(0, 2 * np.pi, n_bins + 1)
    counts, _ = np.histogram(rads, bins=bins)
    midpoints = (bins[:-1] + bins[1:]) / 2
    bars = ax.bar(midpoints, counts, width=bin_width, bottom=0, color=color, alpha=alpha, edgecolor="white", linewidth=0.3)
    ax.set_xticks(np.deg2rad([0, 60, 120, 180, 240, 300]))
    ax.set_xticklabels(["0°", "60°", "120°", "±180°", "-120°", "-60°"])
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_title(title, pad=15, fontweight="bold")
    s = np.sum(np.sin(np.deg2rad(angles_deg)))
    c = np.sum(np.cos(np.deg2rad(angles_deg)))
    n = len(angles_deg)
    circ_mean = np.degrees(np.arctan2(s / n, c / n))
    R = np.sqrt((s / n)**2 + (c / n)**2)
    circ_std = np.degrees(np.sqrt(-2 * np.log(R))) if R > 1e-15 else 180.0
    ax.annotate(f"mean={circ_mean:.1f}°\nstd={circ_std:.1f}°\nn={n:,}", xy=(0.02, 0.98), xycoords="axes fraction", fontsize=7, va="top", ha="left", bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))
    return bars

In [ ]:
display(Markdown("#### 3.1.1 Backbone Torsion Angles (φ, ψ, ω)"))

fig, axes = plt.subplots(1, 3, subplot_kw={"projection": "polar"}, figsize=(14, 4.5))
rose_plot(axes[0], phi, title=f"{DATASET_NAME} — Phi (φ)", color="#2166ac")
rose_plot(axes[1], psi, title=f"{DATASET_NAME} — Psi (ψ)", color="#b2182b")
rose_plot(axes[2], omega, title=f"{DATASET_NAME} — Omega (ω)", color="#1b7837")
fig.suptitle("Backbone Torsion Angle Distributions", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
display(Markdown("#### 3.1.2 Omega Deviation from Planarity (Trans Peptides Only)"))

omega_dev = []
for r in records:
    o = r.get("omega")
    if o is None:
        continue
    pb = r.get("peptide_bond")
    cis = r.get("is_cis")
    if pb is not None:
        is_trans = (pb == "trans")
    elif cis is not None:
        is_trans = (cis == "False") if isinstance(cis, str) else (not cis)
    else:
        is_trans = True
    if is_trans:
        dev = (o % 360) - 180
        omega_dev.append(dev)
omega_dev = np.array(omega_dev)

fig, ax = plt.subplots(figsize=(8, 4))
if len(omega_dev) > 0:
    ax.hist(omega_dev, bins=120, range=(-20, 20), color="#1b7837", alpha=0.8, edgecolor="white", linewidth=0.3)
    ax.axvline(0, color="red", linewidth=0.8, linestyle="--", alpha=0.6, label="Ideal (180°)")
    ax.annotate(f"mean={omega_dev.mean():.3f}°\nstd={omega_dev.std():.3f}°\nn={len(omega_dev):,}", xy=(0.97, 0.95), xycoords="axes fraction", fontsize=9, va="top", ha="right", bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))
    ax.legend(fontsize=8)
ax.set_xlabel("Deviation from ω = 180° (degrees)", fontsize=11)
ax.set_ylabel("Count", fontsize=11)
ax.set_title(f"{DATASET_NAME} — Omega Deviation from Planarity (Trans Only)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
display(Markdown("#### 3.1.3 Sidechain Torsion Angles (χ1–χ4)"))

n_chi = len(chi)
if n_chi > 0:
    chi_colors = ["#e66101", "#5e3c99", "#d95f02", "#7570b3"]
    fig, axes = plt.subplots(1, n_chi, subplot_kw={"projection": "polar"}, figsize=(4.5 * n_chi, 4.5))
    if n_chi == 1:
        axes = [axes]
    for i, (field, values) in enumerate(chi.items()):
        label = field.replace("chi", "χ")
        rose_plot(axes[i], values, title=f"{DATASET_NAME} — {label}", color=chi_colors[i % len(chi_colors)])
    fig.suptitle("Sidechain Torsion Angle Distributions", fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No chi angle data available in this dataset.")

In [ ]:
display(Markdown("### 3.2 Multivariate Distributions\n\n#### 3.2.1 Ramachandran Distributions (\u03c6 \u00d7 \u03c8 by Category)"))

import math

rama_cat_counts = Counter(r.get(RAMA_FIELD) for r in records if r.get(RAMA_FIELD) is not None)
categories = [cat for cat, _ in rama_cat_counts.most_common()]
n_cats = len(categories)
n_cols = 2
n_rows = math.ceil(n_cats / n_cols)

# Scale plot height to fit on one page: max ~9" usable on letter paper
row_height = min(7, 9.0 / n_rows)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, row_height * n_rows))
if n_cats == 1:
    axes = np.array([[axes]])
elif n_rows == 1:
    axes = axes.reshape(1, -1)

for idx, cat in enumerate(categories):
    row, col = divmod(idx, n_cols)
    ax = axes[row, col]
    cat_records = [(r["phi"], r["psi"]) for r in records
                   if r.get(RAMA_FIELD) == cat
                   and r.get("phi") is not None
                   and r.get("psi") is not None]
    if cat_records:
        ph, ps = zip(*cat_records)
        h = ax.hist2d(ph, ps, bins=72, range=[[-180, 180], [-180, 180]],
                      cmap="Blues", cmin=1)
        ax.scatter(ph, ps, s=0.5, alpha=0.1, c="#333333", rasterized=True)
    ax.set_xlim(-180, 180)
    ax.set_ylim(-180, 180)
    ax.set_xlabel("\u03c6 (degrees)", fontsize=10)
    ax.set_ylabel("\u03c8 (degrees)", fontsize=10)
    n_cat = rama_cat_counts[cat]
    ax.set_title(f"{cat} (n={n_cat:,})", fontsize=11, fontweight="bold")
    ax.set_aspect("equal")
    ax.axhline(0, color="gray", linewidth=0.4, alpha=0.5)
    ax.axvline(0, color="gray", linewidth=0.4, alpha=0.5)
    ax.grid(True, alpha=0.2)

# Hide unused subplot(s) if odd number of categories
for idx in range(n_cats, n_rows * n_cols):
    row, col = divmod(idx, n_cols)
    axes[row, col].set_visible(False)

fig.suptitle(f"{DATASET_NAME} \u2014 Ramachandran by {RAMA_FIELD} Category",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\nCategory counts ({RAMA_FIELD}):")
for cat, n in rama_cat_counts.most_common():
    print(f"  {cat:<12s} {n:>8,}  ({100*n/len(records):5.2f}%)")

In [ ]:
display(Markdown("""## 4. Bond Angle Distributions (Linear Statistics)

Bond angles are non-periodic and are analyzed with standard linear
statistics (arithmetic mean, standard deviation).

### 4.1 Tau (N−Cα−C)
"""))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(tau, bins=80, range=(100, 125), color="#4daf4a", alpha=0.8, edgecolor="white", linewidth=0.3)
ax.set_xlabel("τ (degrees)", fontsize=11)
ax.set_ylabel("Count", fontsize=11)
ax.set_title(f"{DATASET_NAME} — Tau (N-Cα-C) Bond Angle Distribution", fontsize=13, fontweight="bold")
ax.annotate(f"mean={tau.mean():.2f}°\nstd={tau.std():.2f}°\nn={len(tau):,}", xy=(0.97, 0.95), xycoords="axes fraction", fontsize=9, va="top", ha="right", bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))
plt.tight_layout()
plt.show()

In [ ]:
display(Markdown("""## 5. References

- Lovell, S. C., Davis, I. W., Arendall, W. B. III, de Bakker, P. I. W.,
  Word, J. M., Prisant, M. G., Richardson, J. S., & Richardson, D. C. (2003).
  Structure validation by Cα geometry: φ, ψ and Cβ deviation.
  *Proteins*, 50(3), 437–450. doi:10.1002/prot.10286

- Lovell, S. C., Word, J. M., Richardson, J. S., & Richardson, D. C. (2000).
  The penultimate rotamer library.
  *Proteins*, 40(3), 389–408. doi:10.1002/1097-0134

- Hintze, B. J., Lewis, S. M., Richardson, J. S., & Richardson, D. C. (2016).
  Molprobity's ultimate rotamer-library distributions.
  *Proteins*, 84(9), 1177–1189. doi:10.1002/prot.25039

- Williams, C. J., Headd, J. J., Moriarty, N. W., Prisant, M. G., Videau, L. L.,
  Deis, L. N., Verma, V., Keedy, D. A., Hintze, B. J., Chen, V. B.,
  Jain, S., Lewis, S. M., Arendall, W. B. III, Snoeyink, J., Adams, P. D.,
  Lovell, S. C., Richardson, J. S., & Richardson, D. C. (2018).
  MolProbity: More and better reference data for improved all-atom structure validation.
  *Protein Science*, 27(1), 293–315. doi:10.1002/pro.3330

- Word, J. M., Lovell, S. C., Richardson, J. S., & Richardson, D. C. (1999).
  Asparagine and glutamine: using hydrogen atom contacts in the choice of
  side-chain amide orientation. *Journal of Molecular Biology*, 285(4), 1735–1747.

- Davis, I. W., Leaver-Fay, A., Chen, V. B., Block, J. N., Kapral, G. J.,
  Wang, X., Murray, L. W., Arendall, W. B. III, Snoeyink, J.,
  Richardson, J. S., & Richardson, D. C. (2007).
  MolProbity: all-atom contacts and structure validation for proteins and nucleic acids.
  *Nucleic Acids Research*, 35(Web Server), W375–W383. doi:10.1093/nar/gkm216

- Edison, A. S. (2001). Linus Pauling and the planar peptide bond.
  *Nature Structural Biology*, 8(3), 201–202. doi:10.1038/84921

---

*Generated by pydangle-biopython v0.5.1 and the richardson-dataset-curation analysis pipeline.*
"""))

In [ ]:
display(Markdown("""---

## Colophon — Document Production Pipeline

This document is maintained as a Jupyter notebook (`analyze_plots.ipynb`) and
converted to distributable formats using a three-stage pipeline:

**Stage 1: Notebook execution** — The parameterized notebook is executed via
`jupyter nbconvert --execute`, which runs all code cells and captures plots
as inline PNG images.

**Stage 2: Markdown export** — The executed notebook is exported to Markdown
via `jupyter nbconvert --to markdown --no-input`, which strips all code cells
and produces a clean `.md` file with embedded image references. This Markdown
file is the primary distributable document — it renders natively on GitHub and
can be read in any text editor.

**Stage 3: PDF generation** — The Markdown is converted to standalone HTML5
via **Pandoc**, then rendered to a print-ready PDF via **WeasyPrint** with a
custom CSS stylesheet (`pipeline/analysis_pdf.css`) providing blue table
headers, sans-serif fonts, and `@page` rules for letter-size layout.

### Files

| File | Purpose |
|------|---------|
| `pipeline/analyze_plots.ipynb` | Notebook template (parameterized) |
| `pipeline/analysis_pdf.css` | Print stylesheet for PDF generation |
| `{dataset}_analysis_plots.ipynb` | Executed notebook (per dataset) |
| `{dataset}_analysis.md` | Markdown export with image references |
| `{dataset}_analysis_files/` | PNG images referenced by the Markdown |
| `{dataset}_analysis.pdf` | Print-ready PDF (via Pandoc + WeasyPrint) |

### Build commands

```bash
# Execute notebook
jupyter nbconvert --to notebook --execute {dataset}_analysis_plots.ipynb

# Export to Markdown (no code)
jupyter nbconvert --to markdown --no-input {dataset}_analysis_plots.ipynb \\
  --output {dataset}_analysis.md

# Convert Markdown to PDF via Pandoc + WeasyPrint
cd {dataset}/
pandoc {dataset}_analysis.md -t html5 --standalone \\
  -o {dataset}_analysis.html
weasyprint {dataset}_analysis.html {dataset}_analysis.pdf \\
  --base-url . --stylesheet ../pipeline/analysis_pdf.css
```

### Prerequisites

```bash
pip install jupyter nbconvert matplotlib numpy
sudo apt install pandoc weasyprint
```

### Design notes

- All section headings and narrative text are rendered via `display(Markdown())`
  in code cells rather than standalone Markdown cells, because WeasyPrint does
  not reliably render Jupyter's native Markdown cell output in PDFs.
- The notebook is parameterized by `DATASET_NAME`, `DATASET_PATH`, and
  `RAMA_FIELD` (cell 0) for reuse across all 5 datasets.
- Torsion angle statistics use circular means (atan2) and circular standard
  deviations (sqrt(-2 ln R̄)). Bond angle statistics use standard linear
  arithmetic. See Section 3 for details.
- The CSS stylesheet is based on the proven `prisant_lab_inventory.css` pattern
  from the tda-project-notes repository.
"""))